# Data Audit Notebook

Sanity checks for SQLite ingestion outputs with a focus on small-table previews and key field quality checks.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path('../data/interim/books.db')
assert DB_PATH.exists(), f'Missing database at {DB_PATH.resolve()}'
conn = sqlite3.connect(DB_PATH)
print('Connected to:', DB_PATH.resolve())

In [ ]:
# 1) List tables
tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    conn
)
tables

In [ ]:
# 2) Row counts per table
counts = []
for t in tables['table_name']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    counts.append({'table_name': t, 'row_count': n})
counts_df = pd.DataFrame(counts).sort_values('table_name').reset_index(drop=True)
counts_df

In [ ]:
# 3) ISBN13 and work_key format checks in openlibrary_enrichment
quality_sql = """
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN isbn13 GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]' THEN 1 ELSE 0 END) AS isbn13_valid_rows,
  SUM(CASE WHEN isbn13 NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]' THEN 1 ELSE 0 END) AS isbn13_invalid_rows,
  SUM(CASE WHEN work_key IS NULL OR TRIM(work_key) = '' THEN 1 ELSE 0 END) AS work_key_blank_rows,
  SUM(CASE WHEN work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key GLOB '/works/*' THEN 1 ELSE 0 END) AS work_key_valid_rows,
  SUM(CASE WHEN work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key NOT GLOB '/works/*' THEN 1 ELSE 0 END) AS work_key_invalid_rows
FROM openlibrary_enrichment
"""
pd.read_sql_query(quality_sql, conn)

In [ ]:
# 4) Show malformed isbn13/work_key rows for inspection
bad_rows_sql = """
SELECT isbn13, work_key, last_error, last_checked_at
FROM openlibrary_enrichment
WHERE isbn13 NOT GLOB '[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]'
   OR (work_key IS NOT NULL AND TRIM(work_key) <> '' AND work_key NOT GLOB '/works/*')
ORDER BY isbn13
"""
pd.read_sql_query(bad_rows_sql, conn)

In [ ]:
# 5) Preview small tables in full, otherwise head(20)
def preview_table(table_name: str, small_threshold: int = 100):
    n = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
    if n <= small_threshold:
        q = f'SELECT * FROM {table_name}'
    else:
        q = f'SELECT * FROM {table_name} LIMIT 20'
    df = pd.read_sql_query(q, conn)
    print(f'\n=== {table_name} (rows={n}) ===')
    return df

for t in tables['table_name']:
    display(preview_table(t))

In [ ]:
# 6) Optional: close connection when done
conn.close()